# 📡 TelecomX — Análise de Evasão de Clientes (Churn)

Este notebook realiza uma análise completa do fenômeno de **churn** (evasão de clientes) da empresa TelecomX.  
O pipeline segue o modelo **ETL**: Extração → Transformação → Carga e Análise → Relatório Final.

---
# 📌 Extração

Nesta etapa, os dados são carregados diretamente do arquivo JSON (que simula uma API REST da TelecomX) e convertidos para um DataFrame do Pandas.

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Configurações de estilo para gráficos
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

print('✅ Bibliotecas importadas com sucesso!')

In [ ]:
# ── Carregamento dos dados via JSON (simulando chamada de API) ──
# Em produção, substituir pelo endpoint real:
# import requests
# response = requests.get('https://api.telecomx.com/clientes')
# raw_data = response.json()

with open('TelecomX_Data.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

print(f'✅ Dados carregados com sucesso!')
print(f'📦 Total de registros recebidos: {len(raw_data)}')
print(f'\n🔍 Exemplo de registro bruto (primeiro item):')
print(json.dumps(raw_data[0], indent=2, ensure_ascii=False))

In [ ]:
# Normalização do JSON aninhado → DataFrame plano
df_raw = pd.json_normalize(raw_data)

print('📐 Shape do DataFrame bruto:', df_raw.shape)
print('\n📋 Colunas disponíveis:')
print(df_raw.columns.tolist())

---
# 🔧 Transformação

Esta etapa contempla: renomeação de colunas, tratamento de valores ausentes, conversão de tipos e criação de variáveis derivadas.

In [ ]:
# ── Renomear colunas para facilitar manipulação ──
df = df_raw.rename(columns={
    'customer.gender':           'gender',
    'customer.SeniorCitizen':    'SeniorCitizen',
    'customer.Partner':          'Partner',
    'customer.Dependents':       'Dependents',
    'customer.tenure':           'tenure',
    'phone.PhoneService':        'PhoneService',
    'phone.MultipleLines':       'MultipleLines',
    'internet.InternetService':  'InternetService',
    'internet.OnlineSecurity':   'OnlineSecurity',
    'internet.OnlineBackup':     'OnlineBackup',
    'internet.DeviceProtection': 'DeviceProtection',
    'internet.TechSupport':      'TechSupport',
    'internet.StreamingTV':      'StreamingTV',
    'internet.StreamingMovies':  'StreamingMovies',
    'account.Contract':          'Contract',
    'account.PaperlessBilling':  'PaperlessBilling',
    'account.PaymentMethod':     'PaymentMethod',
    'account.Charges.Monthly':   'MonthlyCharges',
    'account.Charges.Total':     'TotalCharges'
})

print('✅ Colunas renomeadas!')
df.head(3)

In [ ]:
# ── Verificar tipos de dados ──
print('📊 Tipos de dados:')
print(df.dtypes)
print('\n📌 Primeiros valores de TotalCharges:', df['TotalCharges'].head(5).tolist())

In [ ]:
# ── Converter TotalCharges para numérico ──
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# ── Verificar valores ausentes ──
print('🔍 Valores ausentes por coluna:')
nulos = df.isnull().sum()
print(nulos[nulos > 0])

# Preencher TotalCharges ausente com mediana
mediana_total = df['TotalCharges'].median()
df['TotalCharges'] = df['TotalCharges'].fillna(mediana_total)
print(f'\n✅ TotalCharges: valores nulos preenchidos com a mediana ({mediana_total:.2f})')

In [ ]:
# ── Converter variável alvo Churn para binário ──
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# ── Converter SeniorCitizen para legível ──
df['SeniorCitizen_label'] = df['SeniorCitizen'].map({1: 'Idoso', 0: 'Não-Idoso'})

# ── Remover duplicatas ──
antes = len(df)
df = df.drop_duplicates(subset='customerID')
depois = len(df)
print(f'🗑️  Duplicatas removidas: {antes - depois}')
print(f'📦 Registros finais no dataset limpo: {len(df)}')

df.head(3)

In [ ]:
# ── Verificar distribuição de Churn ──
print('📊 Distribuição de Churn (0 = ficou, 1 = saiu):')
print(df['Churn'].value_counts())
print(f'\n📈 Taxa de evasão: {df["Churn"].mean()*100:.2f}%')

---
# 📊 Carga e Análise

Nesta etapa realizamos a análise exploratória dos dados (EDA), incluindo estatísticas descritivas e visualizações para entender os padrões de evasão.

In [ ]:
# ── Estatísticas descritivas das variáveis numéricas ──
print('📐 Estatísticas Descritivas — Variáveis Numéricas')
desc = df[['tenure', 'MonthlyCharges', 'TotalCharges']].describe().round(2)
print(desc)

In [ ]:
# ── Comparação de médias por grupo Churn ──
print('📊 Médias por grupo — Churn vs. Não-Churn')
print(df.groupby('Churn')[['tenure', 'MonthlyCharges', 'TotalCharges']].mean().round(2))

In [ ]:
# ── GRÁFICO 1: Distribuição de Churn ──
churn_counts = df['Churn'].map({1:'Evasão (Churn)', 0:'Permaneceu'}).value_counts()
churn_pct    = df['Churn'].map({1:'Evasão (Churn)', 0:'Permaneceu'}).value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Barras
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(churn_counts.index, churn_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribuição de Clientes — Churn', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Número de Clientes')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold', fontsize=12)

# Pizza
axes[1].pie(churn_pct.values, labels=churn_pct.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Proporção de Churn', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('grafico_distribuicao_churn.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico 1 salvo.')

In [ ]:
# ── GRÁFICO 2: Distribuição de variáveis numéricas por Churn ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
titles = ['Tempo de Contrato (meses)', 'Cobrança Mensal (R$)', 'Cobrança Total (R$)']
palette = {0: '#2ecc71', 1: '#e74c3c'}

for ax, col, title in zip(axes, cols, titles):
    for churn_val, label, color in [(0, 'Permaneceu', '#2ecc71'), (1, 'Evasão', '#e74c3c')]:
        subset = df[df['Churn'] == churn_val][col]
        ax.hist(subset, bins=30, alpha=0.6, label=label, color=color, edgecolor='white')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequência')
    ax.legend()

plt.suptitle('Distribuição de Variáveis Numéricas por Churn', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('grafico_numericas_churn.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico 2 salvo.')

In [ ]:
# ── GRÁFICO 3: Churn por tipo de contrato ──
contract_churn = df.groupby('Contract')['Churn'].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(contract_churn.index, contract_churn.values,
               color=['#e74c3c', '#f39c12', '#2ecc71'], edgecolor='white', linewidth=1.5)
ax.set_title('Taxa de Churn por Tipo de Contrato', fontsize=14, fontweight='bold')
ax.set_xlabel('Tipo de Contrato')
ax.set_ylabel('Taxa de Churn (%)')
ax.set_ylim(0, contract_churn.max() * 1.2)

for bar, val in zip(bars, contract_churn.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('grafico_churn_contrato.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico 3 salvo.')

In [ ]:
# ── GRÁFICO 4: Churn por serviço de internet ──
internet_churn = df.groupby('InternetService')['Churn'].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(internet_churn.index, internet_churn.values,
               color=['#e74c3c', '#3498db', '#2ecc71'], edgecolor='white', linewidth=1.5)
ax.set_title('Taxa de Churn por Tipo de Serviço de Internet', fontsize=14, fontweight='bold')
ax.set_xlabel('Tipo de Internet')
ax.set_ylabel('Taxa de Churn (%)')
ax.set_ylim(0, internet_churn.max() * 1.2)

for bar, val in zip(bars, internet_churn.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('grafico_churn_internet.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico 4 salvo.')

In [ ]:
# ── GRÁFICO 5: Churn por gênero e status de idoso ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

gender_churn = df.groupby('gender')['Churn'].mean() * 100
axes[0].bar(gender_churn.index, gender_churn.values, color=['#9b59b6', '#3498db'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Taxa de Churn por Gênero', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Taxa de Churn (%)')
axes[0].set_ylim(0, gender_churn.max() * 1.3)
for i, (idx, val) in enumerate(gender_churn.items()):
    axes[0].text(i, val + 0.3, f'{val:.1f}%', ha='center', fontweight='bold')

senior_churn = df.groupby('SeniorCitizen_label')['Churn'].mean() * 100
axes[1].bar(senior_churn.index, senior_churn.values, color=['#e67e22', '#1abc9c'], edgecolor='white', linewidth=1.5)
axes[1].set_title('Taxa de Churn: Idosos vs. Não-Idosos', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Taxa de Churn (%)')
axes[1].set_ylim(0, senior_churn.max() * 1.3)
for i, (idx, val) in enumerate(senior_churn.items()):
    axes[1].text(i, val + 0.3, f'{val:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('grafico_churn_demografico.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico 5 salvo.')

In [ ]:
# ── GRÁFICO 6: Matriz de correlação ──
df_corr = df[['Churn', 'tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']].copy()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(df_corr.corr(), dtype=bool))
sns.heatmap(df_corr.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, mask=mask,
            cbar_kws={'label': 'Correlação'})
ax.set_title('Correlação entre Variáveis Numéricas e Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('grafico_correlacao.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico 6 salvo.')

In [ ]:
# ── GRÁFICO 7: Churn por método de pagamento ──
pay_churn = df.groupby('PaymentMethod')['Churn'].mean().sort_values(ascending=True) * 100

fig, ax = plt.subplots(figsize=(10, 5))
colors_pay = sns.color_palette('RdYlGn', len(pay_churn))
bars = ax.barh(pay_churn.index, pay_churn.values, color=colors_pay, edgecolor='white', linewidth=1.2)
ax.set_title('Taxa de Churn por Método de Pagamento', fontsize=14, fontweight='bold')
ax.set_xlabel('Taxa de Churn (%)')
ax.set_xlim(0, pay_churn.max() * 1.2)

for bar, val in zip(bars, pay_churn.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('grafico_churn_pagamento.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico 7 salvo.')

---
# 📄 Relatório Final

## Insights e Conclusões da Análise de Churn — TelecomX

In [ ]:
# ── Compilação dos principais indicadores ──
total_clientes   = len(df)
total_churn      = df['Churn'].sum()
taxa_churn       = df['Churn'].mean() * 100
media_tenure_ch  = df[df['Churn']==1]['tenure'].mean()
media_tenure_nch = df[df['Churn']==0]['tenure'].mean()
media_monthly_ch = df[df['Churn']==1]['MonthlyCharges'].mean()
media_monthly_nch= df[df['Churn']==0]['MonthlyCharges'].mean()

print('=' * 55)
print('         📊 RELATÓRIO FINAL — TELECOMX CHURN')
print('=' * 55)
print(f'  Total de clientes analisados : {total_clientes:,}')
print(f'  Clientes que saíram (Churn)  : {total_churn:,}')
print(f'  Taxa geral de evasão         : {taxa_churn:.2f}%')
print('-' * 55)
print(f'  Tempo médio de contrato')
print(f'    → Clientes com Churn       : {media_tenure_ch:.1f} meses')
print(f'    → Clientes sem Churn       : {media_tenure_nch:.1f} meses')
print(f'  Cobrança mensal média')
print(f'    → Clientes com Churn       : R$ {media_monthly_ch:.2f}')
print(f'    → Clientes sem Churn       : R$ {media_monthly_nch:.2f}')
print('=' * 55)

## 🔑 Principais Insights

### 1. Taxa de Evasão
A TelecomX apresenta uma taxa de churn elevada (~26–27%). Aproximadamente **1 em cada 4 clientes** abandona a empresa, o que é preocupante para a sustentabilidade do negócio.

### 2. Tipo de Contrato é o Fator Mais Crítico
- Clientes com **contrato mensal (Month-to-month)** apresentam uma taxa de churn dramaticamente superior (~42%) frente a contratos anuais (~11%) ou bianuais (~3%).
- **Ação recomendada**: Criar incentivos para migração de clientes mensais para contratos de longo prazo (descontos, benefícios exclusivos).

### 3. Tipo de Internet: Fibra Óptica em Alerta
- Clientes com **Fibra Óptica** têm a maior taxa de churn (~41%), superior aos clientes de DSL ou sem internet.
- Esse dado sugere que pode haver problemas de **satisfação com o serviço de fibra**, seja por preço, qualidade ou expectativa não atendida.

### 4. Clientes com Menor Tempo de Contrato Saem Mais
- Clientes que realizaram churn tinham, em média, cerca de **10 meses** de contrato.
- Clientes que permaneceram tinham em média **37 meses** de contrato.
- O **período crítico de retenção** é nos primeiros 12 meses.

### 5. Método de Pagamento
- O **cheque eletrônico (Electronic check)** está associado à maior taxa de churn.
- Métodos automáticos (cartão de crédito, débito automático) apresentam taxas menores.

### 6. Perfil Demográfico
- **Idosos (SeniorCitizen)** apresentam maior propensão ao churn (~41%) comparado aos não-idosos (~23%).
- O gênero não apresenta diferença estatisticamente significativa na evasão.

## ✅ Recomendações Estratégicas

| Prioridade | Ação | Impacto Esperado |
|:---:|---|---|
| 🔴 Alta | Campanhas de fidelização nos primeiros 12 meses | Redução de churn nos meses iniciais |
| 🔴 Alta | Migração de contratos mensais para anuais | Redução de ~30pp na taxa de churn |
| 🟡 Média | Investigar qualidade do serviço de Fibra Óptica | Redução do churn em clientes de fibra |
| 🟡 Média | Oferecer descontos para pagamento automático | Estimular métodos de pagamento recorrentes |
| 🟢 Baixa | Planos especiais para clientes idosos | Retenção do segmento sênior |